# 第10章: ラベルなしデータのクラスタリング分析

この Notebook は、原本 `machine-learning-book/ch10/ch10.ipynb` を最新の Python 環境と
`pytest --nbmake` による CI 実行向けに移行したものです。

原本の教育意図を保ちながら、次の点を調整しています。

- `watermark` のような Notebook 拡張依存は外し、標準ライブラリと主要パッケージのみで実行できるようにする
- `AgglomerativeClustering` の API 差分を現行 scikit-learn に合わせる
- k-means、エルボー法、シルエット係数、階層クラスタリング、DBSCAN を CI で継続検証できる軽量構成に整理する


## この Notebook で確認すること

- 原本図版を `src/` 配下から参照できることを確認する
- `make_blobs` に対する k-means クラスタリングと重心推定を再現する
- エルボー法とシルエット係数でクラスタ数の妥当性を評価する
- 階層クラスタリングの距離行列、リンケージ、デンドログラムの流れを確認する
- 半月型データに対して k-means / Agglomerative / DBSCAN を比較する


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

from IPython.display import Image, display
from matplotlib import cm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.metrics import silhouette_samples, silhouette_score


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを見つけられませんでした。")


REPO_ROOT = find_repo_root()
CHAPTER_DIR = REPO_ROOT / "machine-learning-book" / "ch10"
FIGURE_DIR = CHAPTER_DIR / "figures"

assert FIGURE_DIR.exists(), f"図版ディレクトリが見つかりません: {FIGURE_DIR}"

print(f"Python 実行ファイル: {sys.executable}")
print(f"Python バージョン: {platform.python_version()}")
print(f"Matplotlib バックエンド: {matplotlib.get_backend()}")
print(f"Chapter directory: {CHAPTER_DIR}")


In [ ]:
package_versions = pd.DataFrame(
    [
        ("numpy", version("numpy")),
        ("pandas", version("pandas")),
        ("matplotlib", version("matplotlib")),
        ("scikit-learn", version("scikit-learn")),
        ("scipy", version("scipy")),
        ("pytest", version("pytest")),
    ],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版の参照

原本の代表図版を読み込み、移行版 Notebook から読み取り専用サブモジュールのアセットにアクセスできることを確認します。


In [ ]:
for figure_name in ["10_01.png", "10_11.png", "10_16.png"]:
    print(figure_name)
    display(Image(filename=str(FIGURE_DIR / figure_name), width=560))


## k-means クラスタリング

原本と同じ `make_blobs` データセットを使い、3 クラスタの k-means を学習して重心を可視化します。


In [ ]:
X_blobs, y_blobs = make_blobs(
    n_samples=150,
    n_features=2,
    centers=3,
    cluster_std=0.5,
    shuffle=True,
    random_state=0,
)

km = KMeans(
    n_clusters=3,
    init="random",
    n_init=10,
    max_iter=300,
    tol=1e-4,
    random_state=0,
)
y_km = km.fit_predict(X_blobs)

fig, ax = plt.subplots(figsize=(5.2, 4.0))
colors = ["lightgreen", "orange", "lightblue"]
markers = ["s", "o", "v"]
for idx, (color, marker) in enumerate(zip(colors, markers)):
    ax.scatter(
        X_blobs[y_km == idx, 0],
        X_blobs[y_km == idx, 1],
        s=40,
        c=color,
        marker=marker,
        edgecolor="black",
        label=f"Cluster {idx + 1}",
    )
ax.scatter(
    km.cluster_centers_[:, 0],
    km.cluster_centers_[:, 1],
    s=220,
    marker="*",
    c="red",
    edgecolor="black",
    label="Centroids",
)
ax.set_xlabel("Feature 1")
ax.set_ylabel("Feature 2")
ax.legend(loc="best")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

pd.Series(
    {
        "inertia": round(float(km.inertia_), 3),
        "cluster_sizes": y_km.shape[0],
        "centroid_1_x": round(float(km.cluster_centers_[0, 0]), 3),
    }
).to_frame(name="値")


## エルボー法

クラスタ数を 1 から 10 まで変えたときの歪みを確認し、3 クラスタ付近で折れ曲がりがあることを見ます。


In [ ]:
distortions = []
for n_clusters in range(1, 11):
    model = KMeans(
        n_clusters=n_clusters,
        init="k-means++",
        n_init=10,
        max_iter=300,
        random_state=0,
    )
    model.fit(X_blobs)
    distortions.append(model.inertia_)

fig, ax = plt.subplots(figsize=(5.2, 3.4))
ax.plot(range(1, 11), distortions, marker="o", linewidth=2)
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Distortion")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame({"k": range(1, 11), "distortion": np.round(distortions, 3)})


## シルエット係数による評価

3 クラスタと 2 クラスタを比較し、より妥当な分割では平均シルエット係数が高くなることを確認します。


In [ ]:
def plot_silhouette(ax, X: np.ndarray, labels: np.ndarray, title: str) -> float:
    cluster_labels = np.unique(labels)
    silhouette_vals = silhouette_samples(X, labels, metric="euclidean")
    y_ax_lower, y_ax_upper = 0, 0
    yticks = []
    for i, c in enumerate(cluster_labels):
        c_silhouette_vals = silhouette_vals[labels == c]
        c_silhouette_vals.sort()
        y_ax_upper += len(c_silhouette_vals)
        color = cm.jet(float(i) / len(cluster_labels))
        ax.barh(
            range(y_ax_lower, y_ax_upper),
            c_silhouette_vals,
            height=1.0,
            edgecolor="none",
            color=color,
        )
        yticks.append((y_ax_lower + y_ax_upper) / 2.0)
        y_ax_lower += len(c_silhouette_vals)

    silhouette_avg = float(np.mean(silhouette_vals))
    ax.axvline(silhouette_avg, color="red", linestyle="--")
    ax.set_yticks(yticks)
    ax.set_yticklabels(cluster_labels + 1)
    ax.set_ylabel("Cluster")
    ax.set_xlabel("Silhouette coefficient")
    ax.set_title(title)
    return silhouette_avg


km_good = KMeans(n_clusters=3, init="k-means++", n_init=10, max_iter=300, tol=1e-4, random_state=0)
labels_good = km_good.fit_predict(X_blobs)

km_bad = KMeans(n_clusters=2, init="k-means++", n_init=10, max_iter=300, tol=1e-4, random_state=0)
labels_bad = km_bad.fit_predict(X_blobs)

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6), sharey=True)
good_score = plot_silhouette(axes[0], X_blobs, labels_good, "k=3")
bad_score = plot_silhouette(axes[1], X_blobs, labels_bad, "k=2")
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame(
    [
        {"model": "k=3", "silhouette_avg": round(good_score, 4)},
        {"model": "k=2", "silhouette_avg": round(bad_score, 4)},
    ]
)


## 階層クラスタリング

原本と同じ 5 サンプルの小さな表を用い、距離行列、リンケージ行列、デンドログラムの関係を確認します。


In [ ]:
np.random.seed(123)
variables = ["X", "Y", "Z"]
labels_small = ["ID_0", "ID_1", "ID_2", "ID_3", "ID_4"]

X_small = np.random.random_sample((5, 3)) * 10
df_small = pd.DataFrame(X_small, columns=variables, index=labels_small)
row_dist = pd.DataFrame(squareform(pdist(df_small, metric="euclidean")), columns=labels_small, index=labels_small)
row_clusters = linkage(df_small.values, method="complete", metric="euclidean")

display(df_small.round(3))
display(row_dist.round(3))
display(
    pd.DataFrame(
        row_clusters,
        columns=["row label 1", "row label 2", "distance", "no. of items in clust."],
        index=[f"cluster {i + 1}" for i in range(row_clusters.shape[0])],
    ).round(3)
)

fig, ax = plt.subplots(figsize=(6.0, 3.6))
dendrogram(row_clusters, labels=labels_small, ax=ax)
ax.set_ylabel("Euclidean distance")
plt.tight_layout()
plt.show()
plt.close(fig)


## デンドログラム付きヒートマップ

デンドログラムの葉順に並べ替えた行列をヒートマップとして表示し、近いサンプル同士が隣接する様子を確認します。


In [ ]:
fig = plt.figure(figsize=(7.2, 6.4), facecolor="white")
axd = fig.add_axes([0.05, 0.1, 0.2, 0.6])
row_dendr = dendrogram(row_clusters, orientation="left")
axd.set_xticks([])
axd.set_yticks([])
for spine in axd.spines.values():
    spine.set_visible(False)

df_rowclust = df_small.iloc[row_dendr["leaves"][::-1]]
axm = fig.add_axes([0.3, 0.1, 0.55, 0.6])
cax = axm.matshow(df_rowclust, interpolation="nearest", cmap="hot_r")
fig.colorbar(cax)
axm.set_xticks(range(len(df_rowclust.columns)))
axm.set_yticks(range(len(df_rowclust.index)))
axm.set_xticklabels(df_rowclust.columns, rotation=45, ha="left")
axm.set_yticklabels(df_rowclust.index)
plt.show()
plt.close(fig)

df_rowclust.round(3)


## Agglomerative Clustering

scikit-learn 実装でも、クラスタ数を 3 と 2 に変えたときのラベル差を確認します。現行 API に合わせて `metric` を使用します。


In [ ]:
ac3 = AgglomerativeClustering(n_clusters=3, metric="euclidean", linkage="complete")
ac2 = AgglomerativeClustering(n_clusters=2, metric="euclidean", linkage="complete")

labels_ac3 = ac3.fit_predict(X_small)
labels_ac2 = ac2.fit_predict(X_small)

pd.DataFrame(
    {
        "sample": labels_small,
        "agglomerative_k3": labels_ac3,
        "agglomerative_k2": labels_ac2,
    }
)


## 半月型データでのクラスタリング比較

非凸なクラスタ構造を持つ `make_moons` データに対して、k-means、Agglomerative、DBSCAN を比較します。
DBSCAN が形状に沿った分割を作りやすいことを確認します。


In [ ]:
X_moons, _ = make_moons(n_samples=200, noise=0.05, random_state=0)

km_moons = KMeans(n_clusters=2, random_state=0, n_init=10)
labels_km_moons = km_moons.fit_predict(X_moons)

ac_moons = AgglomerativeClustering(n_clusters=2, metric="euclidean", linkage="complete")
labels_ac_moons = ac_moons.fit_predict(X_moons)

dbscan = DBSCAN(eps=0.2, min_samples=5, metric="euclidean")
labels_dbscan = dbscan.fit_predict(X_moons)

fig, axes = plt.subplots(1, 3, figsize=(11.0, 3.4), sharex=True, sharey=True)
for ax, labels_plot, title in [
    (axes[0], labels_km_moons, "K-means"),
    (axes[1], labels_ac_moons, "Agglomerative"),
    (axes[2], labels_dbscan, "DBSCAN"),
]:
    unique_labels = np.unique(labels_plot)
    for label in unique_labels:
        mask = labels_plot == label
        display_label = "Noise" if label == -1 else f"Cluster {label + 1}"
        ax.scatter(
            X_moons[mask, 0],
            X_moons[mask, 1],
            s=28,
            edgecolor="black",
            label=display_label,
        )
    ax.set_title(title)
    ax.set_xlabel("Feature 1")
axes[0].set_ylabel("Feature 2")
axes[2].legend(loc="best")
plt.tight_layout()
plt.show()
plt.close(fig)

pd.DataFrame(
    [
        {"model": "KMeans", "n_clusters_found": len(np.unique(labels_km_moons)), "silhouette": round(silhouette_score(X_moons, labels_km_moons), 4)},
        {"model": "Agglomerative", "n_clusters_found": len(np.unique(labels_ac_moons)), "silhouette": round(silhouette_score(X_moons, labels_ac_moons), 4)},
        {"model": "DBSCAN", "n_clusters_found": len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0), "noise_points": int(np.sum(labels_dbscan == -1))},
    ]
)


## まとめ

この移行版 Notebook では、第10章の中心となる k-means、エルボー法、シルエット評価、
階層クラスタリング、Agglomerative Clustering、DBSCAN を最新の scikit-learn / SciPy API で再構成しました。

原本 `machine-learning-book/` 配下は変更せず、`src/ch10/ch10.ipynb` から読み取り専用図版を参照する構成にしているため、
CI 上でもクラスタリング章の主要ロジックを継続検証できます。
